In [1]:
from __future__ import annotations

import json
from collections import defaultdict
from collections.abc import Iterable
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity
from kebab.utils.io_helpers import resolve_path

In [2]:
# whether to save statistics as JSON files
save_statistics = False

In [ ]:
# (fragments or pairs)
# dataset_path = Path.cwd().parent / "data" / "REBEL" / "linking" / "test" / "rebel_linking_dataset.jsonl"
dataset_path = (
    Path.cwd().parent / "data" / "REBEL" / "entity_generation" / "test" / "rebel_entity_generation_dataset.jsonl"
)

# # entity map
# fragment_to_entity_map_path = (
#     Path.cwd().parent / "data" / "REBEL" / "data" / "linking" / "base" / "rebel_fragment_to_entity_map.jsonl"
# )

# fragment_to_entity_map = defaultdict(list)
# with open(fragment_to_entity_map_path, "r", encoding="utf-8") as f:
#     for line in f:
#         fragment_id, entity_id = json.loads(line)
#         fragment_to_entity_map[fragment_id] = entity_id

# print(f"Loaded entity map for {len(fragment_to_entity_map)} fragments.")

Load the fragments

In [4]:
fragments = []

# we will not be using fragments with no names
fragments_with_no_names = 0


def load_jsonl(file_path: Path) -> Iterable[ResolvedWikidataEntity]:
    """Load a jsonl file containing either ResolvedWikidataEntity objects or tuples of such objects."""
    is_tuples = None

    with open(file_path, encoding="utf-8") as f:
        for line in f:
            if is_tuples is None:
                t = json.loads(line)
                is_tuples = isinstance(t, list)

            if is_tuples:
                left, right = json.loads(line)
                yield ResolvedWikidataEntity.from_dict(left)
                yield ResolvedWikidataEntity.from_dict(right)
            else:
                yield ResolvedWikidataEntity.from_json(line)


seen = set()

for fragment in load_jsonl(dataset_path):
    if not fragment.names:
        fragments_with_no_names += 1
        del fragment
        continue

    prop_str = fragment.property_values_str()

    if prop_str in seen:
        del fragment
        continue

    seen.add(prop_str)

    if not fragment.entity_id and "entity_id" in fragment.metadata:
        fragment.entity_id = fragment.metadata["entity_id"]

    # reduce memory footprint
    if "entity_id" in fragment.metadata:
        del fragment.metadata["entity_id"]

    if "fragment_id" in fragment.metadata:
        del fragment.metadata["fragment_id"]

    if "fragment_ids" in fragment.metadata:
        del fragment.metadata["fragment_ids"]

    fragment.evidence_map = None  # type: ignore
    fragment.source_ids = None  # type: ignore

    fragments.append(fragment)

print(f"Loaded {len(fragments):,d} fragments, ignored {fragments_with_no_names:,d} fragments with no names")

Loaded 5,108 fragments, ignored 0 fragments with no names


Example fragment

In [5]:
fragments[0]

ResolvedWikidataEntity(entity_id='Q100048', properties=defaultdict(<class 'list'>, {'name': ['Concordia sulla Secchia', 'Concordia'], 'instance of': ['comune'], 'located in the administrative territorial entity': ['Province of Modena'], 'country': ['Italian'], 'shares border with': ['Mirandola', 'Moglia', 'Novi di Modena', 'Quistello', 'San Giacomo delle Segnate', 'San Giovanni del Dosso', 'San Possidonio']}), source_ids=None, evidence_map=None, metadata={'merge_count': 7, 'type': ['commune of Italy'], 'title': 'Concordia sulla Secchia'})

In [22]:
filtered = []

for fragment in fragments:
    if "point in time" in fragment.properties:
        continue

    names = set(fragment.names)
    found = False
    for prop_name, prop_values in fragment.properties.items():
        if found:
            break

        if prop_name == "name":
            continue

        for v in prop_values:
            if v in names:
                filtered.append(fragment)
                print(f"'{v}' found in '{prop_name}':")
                # print("Names: " + str(fragment.names))
                print(dict(fragment.properties))
                print(fragment.metadata)
                print()
                found = True

                break

print(f"Filtered down to {len(filtered):,d} fragments where a property value matches a name")

'Ostia' found in 'location':
{'name': ['Ostia', 'Ostia Antica', 'Ostian', 'ancient Ostia'], 'located in the administrative territorial entity': ['Rome', 'Rome, Italy'], 'location': ['Ostia'], 'instance of': ['archaeological sites'], 'country': ['Italy', 'Italian']}
{'merge_count': 11, 'type': ['archaeological site', 'ancient city'], 'title': 'San Giovanni di Posada'}

'Los Angeles' found in 'headquarters location':
{'name': ['California State University, Los Angeles', 'Los Angeles State', 'California State University at Los Angeles', 'Cal State Los Angeles', 'Los Angeles State College', 'California State Los Angeles', 'Luckman Fine Arts Complex', 'Los Angeles', 'Cal State LA', 'CSULA', 'California State University Los Angeles', 'Los Angeles State College of Applied Arts and Sciences', 'California State College'], 'headquarters location': ['Los Angeles, California', 'Los Angeles'], 'parent organization': ['CSU', 'California State University']}
{'merge_count': 17, 'type': ['university', 

In [8]:
# compute counts of property occurrence and entity types of the fragments
property_counts = defaultdict(int)
type_counts = defaultdict(int)

for fragment in fragments:
    for prop_name, prop_value in fragment.properties.items():
        if prop_value:
            property_counts[prop_name] += 1

    if fragment.wikidata_type:
        for ent_type in fragment.wikidata_type:
            type_counts[ent_type] += 1

Top properties by occurrence in the fragments
---

In [9]:
df = (
    pd.DataFrame(property_counts.items(), columns=["property", "count"])
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)

if save_statistics:
    df.to_csv("property_occurrence.csv", index=False)

df[:20]

,property,count
0,name,5108
1,located in the administrative territorial entity,1342
2,country,1277
3,instance of,967
4,date of birth,786
5,sport,457
6,point in time,445
7,inception,380
8,date of death,355
9,part of,327


Top entity types of the fragments
---

In [10]:
df = (
    pd.DataFrame(type_counts.items(), columns=["type", "count"])
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)

if save_statistics:
    df.to_csv("type_occurrence.csv", index=False)

df[:20]

,type,count
0,human,1098
1,sports series,177
2,commune of Italy,145
3,river,73
4,album,70
5,film,70
6,American football team season,68
7,human settlement,55
8,commune of France,51
9,corporation,50


Fragments per entity
---

In [11]:
entity_ids = [fragment.entity_id for fragment in fragments]
counts = pd.Series(entity_ids).value_counts().value_counts().sort_index()

fig = px.bar(counts, x=counts.index.astype(str), y=counts.values, title="Fragments per entity", text=counts.values)
fig.update_layout(xaxis_title="", yaxis_title="Count", width=800, height=600)
fig.show()

avg_fragments_per_entity = pd.Series(entity_ids).value_counts().mean()
print(f"Average fragments per entity: {avg_fragments_per_entity:.2f}")

Average fragments per entity: 1.03


Fragments sizes (if merged)
---

In [12]:
merge_counts = [fragment.metadata["merge_count"] if "merge_count" in fragment.metadata else 1 for fragment in fragments]

counts = pd.Series(merge_counts).value_counts().sort_index()
fig = px.bar(
    counts, x=counts.index.astype(str), y=counts.values, title="Fragment sizes (if merged)", text=counts.values
)
fig.update_layout(xaxis_title="Fragment size", yaxis_title="Count", width=800, height=600)
fig.show()

Properties overlap
---

In [13]:
# for each property how often that two distinct fragments have (1) a value for this property, and (2) the same value this property
property_value_counts = defaultdict(lambda: defaultdict(int))
entity_value_counts = defaultdict(int)

for entity_id in fragments:
    for prop_name, values in entity_id.properties.items():
        entity_value_counts[prop_name] += 1
        for value in values:
            property_value_counts[prop_name][value] += 1

entity_count = len(fragments)

rows = []
for prop_name, value_counts in property_value_counts.items():
    arr = np.array(list(value_counts.values()))
    ent_val_count = entity_value_counts[prop_name]
    ent_probs = arr / entity_count
    cond_ent_probs = arr / ent_val_count
    prob = (ent_probs**2).sum()
    cond_prob = (cond_ent_probs**2).sum()
    ent_fraction = ent_val_count / entity_count
    rows.append((prop_name, len(value_counts), prob, cond_prob, ent_fraction))

overlap_df = pd.DataFrame(
    rows, columns=["property", "distinct_value_count", "overlap_prob", "cond_overlap_prob", "entities_fraction"]
)
overlap_df = overlap_df.sort_values("overlap_prob", ascending=False)
overlap_df.head(100)

,property,distinct_value_count,overlap_prob,cond_overlap_prob,entities_fraction
3,country,306,0.001540,0.024648,0.250000
1,instance of,652,0.000996,0.027781,0.189311
7,sport,105,0.000915,0.114307,0.089468
0,name,13094,0.000596,0.000596,1.000000
2,located in the administrative territorial entity,1317,0.000134,0.001937,0.262725
...,...,...,...,...,...
322,student,31,0.000001,0.861111,0.001175
131,screenwriter,29,0.000001,0.050347,0.004699
109,candidate,28,0.000001,0.571429,0.001370
26,highest point,27,0.000001,0.043200,0.004894
